# 02 — Three operational approaches (all free/open)
1. **Local open model** — Ollama / llama.cpp serving a model such as Laya, Kev or Tev1.
2. **Free hosted route** — a free tier endpoint, used only if you want one.
3. **Training-free logit reading** — open methods (LitJev, SemIf, AnyJev) read option probabilities from any open model without fine-tuning.

The offline mock stands in for any of them so this notebook runs everywhere.


In [ ]:
from decision_qa import load_config, SystemOneClient, MockClient

local_cfg = load_config('configs/local_ollama.yaml')
hosted_cfg = load_config('configs/hosted.yaml')
print('local :', local_cfg.backend, local_cfg.base_url, local_cfg.model)
print('hosted:', hosted_cfg.backend, hosted_cfg.base_url, hosted_cfg.model)


## Compare on identical inputs
We time each client on the same questions. A live call is attempted only when a local server is actually running; otherwise the mock is used.


In [ ]:
import time
from decision_qa import NoulQuestion, ChoiceQuestion

state = {'context': 'The context says the treatment helped.', 'question': 'Did it help?'}
qs = [
    NoulQuestion(label='helped', statement='The treatment helped.', gold=True),
    ChoiceQuestion(label='verdict', options=['yes', 'no', 'unclear'], gold_index=0),
]

def timed(client, label):
    t0 = time.perf_counter()
    results = client.decision(state, qs)
    dt = (time.perf_counter() - t0) * 1000
    print(f'{label:12s} {dt:7.1f} ms', {r.question_label: round(max(r.probabilities.values()), 3) for r in results})

timed(MockClient(seed=0), 'mock')
try:
    timed(SystemOneClient(local_cfg), 'local')
except Exception as e:
    print('local endpoint not running:', type(e).__name__, '- start Ollama to try it')


## Trade-offs
| Approach | Cost | Setup | Best for |
|---|---|---|---|
| Local open model | free | download weights | privacy, offline |
| Free hosted route | free tier | free account | no local hardware |
| Logit reading | free | open model | fast prototypes |

Open implementations to explore: [LitJev](https://github.com/zhengxuyu/litjev), [SemIf](https://github.com/TheoLeeCJ/SemIf-OpenJev), [AnyJev](https://github.com/nokia-applied-research/AnyJev).
